# Lab 3 — Phân tích cảm xúc với Transformers

**Jupyter Notebook · Mã nguồn, phương pháp và kết quả thực nghiệm**

Notebook là bản trình bày chính của Lab 3. Các bước xử lý dữ liệu, tính metric, quản lý checkpoint và huấn luyện được định nghĩa trực tiếp trong các ô code. `Lab 3/lab3.py` cung cấp các tiện ích lưu trữ, báo cáo và kiểm tra nguồn dùng chung; các định nghĩa cốt lõi được kiểm tra đồng nhất theo AST.

| Hạng mục | Phạm vi |
| --- | --- |
| Exercise 1 | Phân tích cảm xúc đa ngôn ngữ bằng model pretrained, 5 nhãn |
| Exercise 2 | Fine-tune DistilBERT trên IMDB tiếng Anh, 2 nhãn |
| Thực nghiệm full | 20.000 train / 5.000 validation / 25.000 test; 3 epochs |
| Bằng chứng | Metric, loss curve, confusion matrix, 5 checkpoint và manifest SHA-256 |

**Cách sử dụng:** mở notebook, chọn kernel **Python (Lab 3)**, chọn **Run All**. Giữ thư mục `Lab 3/` đi kèm để cung cấp đặc tả, tiện ích và artifact. Các run đã hoàn tất được kiểm hash trước khi tái sử dụng; notebook không huấn luyện full lặp để chọn kết quả test thuận lợi.

## Nguồn và phạm vi

- [Đặc tả](Lab%203/docs/lab_3_description.md), [thiết kế triển khai](Lab%203/docs/implementation.md).
- [Dữ liệu](Lab%203/docs/data.md), [model](Lab%203/docs/model.md), [training](Lab%203/docs/training.md), [evaluation](Lab%203/docs/evaluation.md), [demo](Lab%203/docs/demo.md).
- [Khai báo AI](Lab%203/docs/AI_Usage.md), [tài liệu chính thức](Lab%203/docs/sources.md), [truy vết](Lab%203/docs/traceability.md).

Mỗi ô code khai nguồn Markdown. Kết quả chỉ được trình bày từ run đã thực thi; dữ liệu test không tham gia lựa chọn checkpoint.

## 1. Môi trường và kiểm tra tính nhất quán

Môi trường đã được cài theo `Lab 3/README.md`. Trên môi trường mới, cài requirements và đăng ký kernel trước khi chạy. `INSTALL_DEPENDENCIES=False` giúp tránh cài lại package khi Run All.

Kiểm tra AST dưới đây xác nhận các định nghĩa cốt lõi hiển thị trong notebook khớp với phiên bản CLI. Khi thay đổi thuật toán, cập nhật đặc tả và hai điểm thực thi cùng nhau.

In [ ]:
# Sources: Lab 3/docs/implementation.md; Lab 3/docs/sources.md; Lab 3/docs/AI_Usage.md
from __future__ import annotations
import argparse
import contextlib
import csv
import hashlib
import importlib.metadata
import json
import math
import platform
import shutil
import sys
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import psutil
import torch
from datasets import DatasetDict, load_dataset
from huggingface_hub import HfApi
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainerCallback,
                          TrainingArguments, pipeline, set_seed)

import os, ast, html, subprocess
from IPython.display import display, Markdown, Image
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
ROOT = next((p.resolve() for p in [Path('Lab 3'), Path('.')]
             if (p/'docs/implementation.md').is_file() and (p/'lab3.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Cần đặt notebook cùng thư mục Lab 3/. Xem README.md để chuẩn bị môi trường.')
NOTEBOOK_PATH = ROOT.parent/'Lab_3 (1).ipynb'
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements.txt')], check=True)
sys.path.insert(0, str(ROOT))
import lab3 as support
from lab3 import (utc_now, sha256, write_json, fresh_directory, stratified_indices,
                  environment, plot_results, render_report, verify_artifacts, Tee)
SOURCES = ["docs/implementation.md", "docs/AI_Usage.md", "docs/data.md",
           "docs/model.md", "docs/training.md", "docs/evaluation.md", "docs/demo.md"]
BASE_MODEL = "distilbert/distilbert-base-uncased"
INFERENCE_MODEL = "tabularisai/multilingual-sentiment-analysis"
DATASET = "stanfordnlp/imdb"
DEMO_TEXTS = [
    "This movie was an absolute masterpiece! The acting and plot were brilliant.",
    "Waste of time. The acting was terrible and the storyline made no sense.",
    "It was an average film, neither great nor awful.",
]

module_ast = ast.parse((ROOT/'lab3.py').read_text(encoding='utf-8'))
canonical = {n.name:n for n in module_ast.body if isinstance(n,(ast.FunctionDef,ast.ClassDef))}
notebook_document = json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
matched_symbols = []
for cell in notebook_document['cells']:
    names = cell.get('metadata',{}).get('implementation_symbols',[])
    if not names:
        continue
    definitions = {n.name:n for n in ast.parse(''.join(cell['source'])).body
                   if isinstance(n,(ast.FunctionDef,ast.ClassDef))}
    for name in names:
        if ast.dump(definitions[name],include_attributes=False) != ast.dump(canonical[name],include_attributes=False):
            raise ValueError(f'Định nghĩa {name} khác phiên bản CLI; cần đồng bộ đặc tả và mã nguồn.')
        matched_symbols.append(name)
print(f'Đồng nhất {len(matched_symbols)} định nghĩa cốt lõi: PASS')
print('Thiết bị:', environment()['device'])
print('Python:', environment()['python'], '| PyTorch:', environment()['versions']['torch'])


## 2. Cấu hình thực nghiệm

Cấu hình full giữ cố định trước đánh giá test: seed 42, độ dài tối đa 256 token, batch 16, AdamW với learning rate 2e-5, weight decay .01 và tối đa 3 epochs. CUDA dùng FP16. Smoke chạy 2 bước optimizer trên dữ liệu rút gọn; không dùng smoke để đánh giá KPI chất lượng.

In [ ]:
# Sources: Lab 3/docs/implementation.md; Lab 3/docs/training.md
@dataclass
class RunConfig:
    """Sources: docs/implementation.md §1–2; docs/training.md."""
    mode: str = "full"
    seed: int = 42
    max_length: int = 256
    batch_size: int = 16
    learning_rate: float = 2e-5
    epochs: int = 3
    weight_decay: float = 0.01

    def __post_init__(self):
        if self.mode not in {"smoke", "quick", "full"}:
            raise ValueError("mode must be smoke, quick or full")


def show_table(headers, rows):
    def escape(value):
        return html.escape(str(value)).replace('|', '&#124;').replace(chr(10), '<br>')
    text = '| ' + ' | '.join(map(escape,headers)) + ' |' + chr(10)
    text += '| ' + ' | '.join(['---']*len(headers)) + ' |' + chr(10)
    text += chr(10).join('| ' + ' | '.join(escape(v) for v in row) + ' |' for row in rows)
    display(Markdown(text))

CONFIG = RunConfig(mode='full')
show_table(['Tham số','Giá trị'], list(asdict(CONFIG).items()))


## 3. Exercise 1 — pretrained sentiment analysis

Module A dùng `tabularisai/multilingual-sentiment-analysis`. Tokenizer biểu diễn câu thành token/subword; pipeline trả nhãn có xác suất cao nhất trong 5 lớp. Module này không được đánh giá như bài IMDB nhị phân.

Các định nghĩa dưới đây chạy trực tiếp trong kernel. Nếu artifact Module A đã có, notebook kiểm hash và trình bày kết quả ghi nhận; nếu chưa có, gọi model thật và lưu nguồn.

In [ ]:
# Sources: Lab 3/docs/model.md; Lab 3/docs/demo.md; Lab 3/docs/AI_Usage.md
def validate_texts(texts):
    """R-01/R-08; docs/demo.md: actionable errors instead of empty input."""
    if not texts or any(not isinstance(t, str) or not t.strip() for t in texts):
        raise ValueError("Provide one or more non-empty text strings")
    return texts


def infer_pretrained(texts):
    """R-01; docs/model.md §4: multilingual five-class Module A."""
    validate_texts(texts)
    revision = HfApi().model_info(INFERENCE_MODEL).sha
    tokenizer = AutoTokenizer.from_pretrained(INFERENCE_MODEL, revision=revision)
    classifier = pipeline("text-classification", model=INFERENCE_MODEL,
                          revision=revision, tokenizer=tokenizer,
                          device=0 if torch.cuda.is_available() else -1)
    results = classifier(texts, truncation=True, max_length=512)
    return {"model": INFERENCE_MODEL, "model_revision": revision,
            "predictions": [{"text": text, "tokens": tokenizer.tokenize(text),
                              **result} for text, result in zip(texts, results)]}


A_DIR = ROOT/'outputs/module-a'
if (A_DIR/'inference.json').is_file():
    verify_artifacts(A_DIR)
    a_results = json.loads((A_DIR/'inference.json').read_text(encoding='utf-8'))
else:
    A_DIR = fresh_directory(A_DIR)
    a_results = infer_pretrained(['Bộ phim này rất hay!', 'The movie was boring.'])
    write_json(A_DIR/'inference.json', a_results)
    shutil.copyfile(NOTEBOOK_PATH,A_DIR/'notebook_source.ipynb')
    support.record_provenance(A_DIR,'Notebook: Module A infer_pretrained')
show_table(['Văn bản','Tokens','Nhãn','Score'],
           [(row['text'],' / '.join(row['tokens']),row['label'],f"{row['score']:.6f}")
            for row in a_results['predictions']])


## 4. Exercise 2 — dữ liệu và tiền xử lý

IMDB gồm hai split chính thức train/test. Tập train được chia phân tầng 80/20 thành 20.000 train và 5.000 validation; test giữ đủ 25.000 mẫu. Seed cố định và hash index giúp tái lập split. Chỉ tập train tính gradient; validation chọn model; test chỉ đo cuối.

Tokenizer cắt tại 256 token. Padding được thực hiện động theo batch. `prepare_data` chỉ nạp dữ liệu khi pipeline được chạy mới; xem bảng split của run ở phần kết quả.

In [ ]:
# Sources: Lab 3/docs/data.md; Lab 3/docs/implementation.md
def prepare_data(config, tokenizer, dataset_revision):
    """R-02/R-03; docs/data.md and docs/implementation.md §2."""
    train, test = load_dataset(DATASET, revision=dataset_revision,
                               split=["train", "test"])
    train_idx, val_idx = train_test_split(np.arange(len(train)), test_size=0.2,
                                          stratify=train["label"],
                                          random_state=config.seed)
    sizes = {"smoke": (32, 16, 16), "quick": (1000, 200, 200),
             "full": (len(train_idx), len(val_idx), len(test))}[config.mode]
    train_idx = train_idx[stratified_indices(np.asarray(train["label"])[train_idx],
                                             sizes[0], config.seed)]
    val_idx = val_idx[stratified_indices(np.asarray(train["label"])[val_idx],
                                         sizes[1], config.seed)]
    test_idx = stratified_indices(test["label"], sizes[2], config.seed)
    if set(train_idx) & set(val_idx):
        raise AssertionError("Train and validation overlap")
    raw = DatasetDict(train=train.select(train_idx), validation=train.select(val_idx),
                      test=test.select(test_idx))
    split_info = {}
    for name, indices, official in [("train", train_idx, "train"),
                                     ("validation", val_idx, "train"),
                                     ("test", test_idx, "test")]:
        split_info[name] = {
            "official_split": official, "size": len(raw[name]),
            "class_counts": np.bincount(raw[name]["label"], minlength=2).tolist(),
            "indices_sha256": hashlib.sha256(np.asarray(indices, dtype="<i8").tobytes()).hexdigest(),
            "fingerprint": raw[name]._fingerprint,
        }

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=config.max_length)

    encoded = raw.map(tokenize, batched=True, remove_columns=["text"])
    return raw, encoded, split_info


## 5. Metrics và cách đánh giá

Accuracy đo tỷ lệ đúng. Precision/recall/binary F1 tập trung lớp POSITIVE; macro F1 tính F1 cho từng lớp rồi lấy trung bình, nên không thể thay thế hai giá trị cho nhau. `zero_division=0` xử lý trường hợp chưa dự đoán được một lớp trong smoke.

In [ ]:
# Sources: Lab 3/docs/evaluation.md; Lab 3/docs/sources.md
def compute_metrics(prediction):
    """R-07; docs/evaluation.md: binary and macro are distinct metrics."""
    logits, labels = prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted = np.argmax(logits, axis=-1)
    precision, recall, binary_f1, _ = precision_recall_fscore_support(
        labels, predicted, average="binary", zero_division=0)
    _, _, macro_f1, _ = precision_recall_fscore_support(
        labels, predicted, labels=[0, 1], average="macro", zero_division=0)
    return {"accuracy": float(accuracy_score(labels, predicted)),
            "precision": float(precision), "recall": float(recall),
            "f1_binary": float(binary_f1), "f1_macro": float(macro_f1)}


## 6. Checkpoint và early stopping

Sau mỗi epoch, lưu `best_val_loss` khi loss nhỏ hơn trước và `best_val_acc` khi accuracy cao hơn trước. `best_val_pause` giữ model tại lần validation cuối. `final_epoch` lưu trước khi nạp best; `best` chọn theo validation accuracy.

Counter dừng tăng nếu accuracy không cải thiện **hơn 0,01 điểm tuyệt đối**; dừng khi counter đạt 2. `early_stopping_triggered` biểu thị điều kiện kích hoạt; `early_stopped` chỉ true khi lịch thực sự bị rút ngắn. Điều kiện tại đúng epoch cuối không được trình bày là dừng trước lịch.

In [ ]:
# Sources: Lab 3/docs/training.md; Lab 3/docs/implementation.md
@dataclass
class StopPolicy:
    """R-06; docs/implementation.md §3: absolute accuracy delta > .01."""
    patience: int = 2
    threshold: float = 0.01
    best_accuracy: float = -math.inf
    best_loss: float = math.inf
    stale: int = 0

    def observe(self, accuracy, loss):
        if not math.isfinite(accuracy) or not math.isfinite(loss):
            raise ValueError("Non-finite validation metric")
        accuracy_improved = accuracy > self.best_accuracy
        loss_improved = loss < self.best_loss
        meaningful = accuracy - self.best_accuracy > self.threshold + 1e-12
        self.stale = 0 if meaningful else self.stale + 1
        self.best_accuracy = max(self.best_accuracy, accuracy)
        self.best_loss = min(self.best_loss, loss)
        return accuracy_improved, loss_improved, self.stale >= self.patience


class CheckpointCallback(TrainerCallback):
    """R-05/R-06/R-07. Save inference checkpoints; track sampled resources."""
    def __init__(self, output, tokenizer):
        self.output = Path(output)
        self.tokenizer = tokenizer
        self.policy = StopPolicy()
        self.history = []
        self.gradient_norms = []
        self.epoch_start = None
        self.peak_rss = 0
        self.early_stopped = False

    def sample_resources(self):
        self.peak_rss = max(self.peak_rss, psutil.Process().memory_info().rss)

    def save(self, model, name):
        destination = self.output / "checkpoints" / name
        model.save_pretrained(destination, safe_serialization=True)
        self.tokenizer.save_pretrained(destination)

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start = time.perf_counter()
        self.sample_resources()

    def on_step_end(self, args, state, control, **kwargs):
        self.sample_resources()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "grad_norm" in logs:
            norm = float(logs["grad_norm"])
            if not math.isfinite(norm):
                raise RuntimeError("Non-finite gradient norm")
            self.gradient_norms.append(norm)
        if logs and "loss" in logs and not math.isfinite(float(logs["loss"])):
            raise RuntimeError("Non-finite training loss")

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        # Ignore later standalone validation/test passes: this callback is train-only.
        if metrics is None or "eval_accuracy" not in metrics:
            return control
        acc, loss = float(metrics["eval_accuracy"]), float(metrics["eval_loss"])
        improved_acc, improved_loss, stop = self.policy.observe(acc, loss)
        if improved_acc:
            self.save(model, "best_val_acc")
        if improved_loss:
            self.save(model, "best_val_loss")
        self.save(model, "best_val_pause")
        # A condition reached on the scheduled final step did not shorten training.
        self.early_stopped = bool(stop and state.global_step < state.max_steps)
        if self.early_stopped:
            control.should_training_stop = True
        self.sample_resources()
        row = {"epoch": float(state.epoch), "step": state.global_step,
               "val_loss": loss, "val_accuracy": acc,
               "val_f1_macro": float(metrics["eval_f1_macro"]),
               "val_f1_binary": float(metrics["eval_f1_binary"]),
               "val_precision": float(metrics["eval_precision"]),
               "val_recall": float(metrics["eval_recall"]),
               "epoch_wall_seconds": time.perf_counter() - self.epoch_start,
               "best_val_acc": improved_acc, "best_val_loss": improved_loss,
               "stale_evaluations": self.policy.stale,
               "early_stopping_triggered": stop, "early_stopped": self.early_stopped}
        self.history.append(row)
        write_json(self.output / "epoch_history.json", {"epochs": self.history})
        print("EPOCH " + json.dumps(row), flush=True)
        return control


## 7. Nạp model và đo latency

DistilBERT của Module B có hai nhãn NEGATIVE/POSITIVE. Hàm demo nạp lại checkpoint từ đĩa, chạy `eval()` và `inference_mode()`. Không có lớp NEUTRAL trong bài nhị phân.

Latency được đo với batch 1, 3 lần warm-up và 20 lần đo, gồm tokenize/transfer/forward; đồng bộ CUDA trước/sau. Kết quả chỉ có ý nghĩa với thiết bị và độ dài đầu vào được ghi nhận.

In [ ]:
# Sources: Lab 3/docs/demo.md; Lab 3/docs/evaluation.md; Lab 3/docs/model.md
def load_demo(checkpoint, texts):
    """R-08; always reload serialized model and tokenizer from disk."""
    validate_texts(texts)
    checkpoint = Path(checkpoint)
    if not (checkpoint / "config.json").is_file():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint}")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, local_files_only=True).to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    results = []
    with torch.inference_mode():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True,
                               max_length=256).to(model.device)
            probs = model(**inputs).logits.softmax(-1)[0]
            label_id = int(probs.argmax())
            results.append({"text": text, "label": model.config.id2label[label_id],
                            "score": float(probs[label_id])})
    return model, tokenizer, results


def measure_latency(model, tokenizer, text, repetitions=20):
    """R-07; docs/implementation.md §5: end-to-end tokenization + forward."""
    def sync():
        if model.device.type == "cuda":
            torch.cuda.synchronize()

    timings = []
    model.eval()
    with torch.inference_mode():
        for i in range(repetitions + 3):
            sync()
            start = time.perf_counter()
            inputs = tokenizer(text, return_tensors="pt", truncation=True,
                               max_length=256).to(model.device)
            model(**inputs).logits.softmax(-1)
            sync()
            if i >= 3:
                timings.append((time.perf_counter() - start) * 1000)
    return {"mean_ms": float(np.mean(timings)), "p50_ms": float(np.median(timings)),
            "p95_ms": float(np.percentile(timings, 95)), "repetitions": repetitions,
            "warmup": 3, "batch_size": 1, "token_count": inputs["input_ids"].shape[1],
            "includes_tokenization_and_transfer": True}


## 8. Pipeline fine-tuning trực tiếp trong notebook

Backbone `distilbert-base-uncased` được gắn head 2 lớp và fine-tune toàn bộ tham số. Logits đi trực tiếp vào cross entropy; softmax dùng khi suy luận. AdamW, clipping và mixed precision được cấu hình trong `TrainingArguments`.

Pipeline dưới đây kết nối dữ liệu → Trainer → callback → lưu checkpoint → nạp best → validation/test → báo cáo. Các tiện ích lưu JSON, vẽ hình và truy nguồn dùng chung với module; thuật toán huấn luyện được hiển thị và thực thi trong ô code này.

In [ ]:
# Sources: Lab 3/docs/implementation.md; Lab 3/docs/training.md; Lab 3/docs/evaluation.md
def train_run(config, output):
    """Sources: docs/implementation.md R-02..R-09; docs/training.md."""
    set_seed(config.seed)
    torch.set_num_threads(min(8, torch.get_num_threads()))
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    env = environment()
    model_revision = HfApi().model_info(BASE_MODEL).sha
    dataset_revision = HfApi().dataset_info(DATASET).sha
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=model_revision)
    raw, encoded, splits = prepare_data(config, tokenizer, dataset_revision)
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, revision=model_revision, num_labels=2,
        id2label={0: "NEGATIVE", 1: "POSITIVE"}, label2id={"NEGATIVE": 0, "POSITIVE": 1})
    callback = CheckpointCallback(output, tokenizer)
    write_json(output / "run_config.json", {"config": asdict(config), "splits": splits,
               "environment": env, "model": BASE_MODEL, "model_revision": model_revision,
               "dataset": DATASET, "dataset_revision": dataset_revision})
    args = TrainingArguments(
        output_dir=str(output / "trainer"), eval_strategy="epoch", save_strategy="no",
        learning_rate=config.learning_rate, per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size, num_train_epochs=config.epochs,
        max_steps=2 if config.mode == "smoke" else -1, weight_decay=config.weight_decay,
        max_grad_norm=1.0, optim="adamw_torch", fp16=torch.cuda.is_available(),
        logging_steps=1 if config.mode == "smoke" else 50, logging_first_step=True,
        report_to="none", seed=config.seed, data_seed=config.seed,
        dataloader_num_workers=0, dataloader_pin_memory=torch.cuda.is_available(),
        disable_tqdm=True, log_level="error", save_safetensors=True)
    trainer = Trainer(model=model, args=args, train_dataset=encoded["train"],
                       eval_dataset=encoded["validation"], processing_class=tokenizer,
                       data_collator=DataCollatorWithPadding(tokenizer,
                           pad_to_multiple_of=8 if torch.cuda.is_available() else None),
                       compute_metrics=compute_metrics, callbacks=[callback])
    print("RUN " + json.dumps({"config": asdict(config), "splits": splits,
                                 "environment": env}), flush=True)
    start = time.perf_counter()
    training = trainer.train()
    train_seconds = time.perf_counter() - start
    if not callback.history or not callback.gradient_norms or not any(n > 0 for n in callback.gradient_norms):
        raise RuntimeError("No validation or finite nonzero gradient evidence")
    callback.save(trainer.model, "final_epoch")
    shutil.copytree(output / "checkpoints/best_val_acc", output / "checkpoints/best")
    write_json(output / "checkpoint_policy.json", {"policy": asdict(callback.policy),
               "early_stopped": callback.early_stopped, "best_source": "best_val_acc",
               "early_stopping_triggered": callback.policy.stale >= callback.policy.patience,
               "best_val_pause_means": "last validation model; see early_stopped flag"})
    write_json(output / "trainer_history.json", {"history": trainer.state.log_history})
    # Free optimizer + old model before reloading best; final_epoch is already saved.
    trainer.remove_callback(callback)
    trainer.optimizer = None
    trainer.lr_scheduler = None
    trainer.model.to("cpu")
    model, tokenizer, demo_results = load_demo(output / "checkpoints/best", DEMO_TEXTS)
    evaluator = Trainer(model=model, args=args, processing_class=tokenizer,
                        eval_dataset=encoded["validation"],
                        data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
    validation = evaluator.evaluate(encoded["validation"], metric_key_prefix="validation")
    prediction = evaluator.predict(encoded["test"], metric_key_prefix="test")
    predicted = prediction.predictions.argmax(-1)
    matrix = confusion_matrix(prediction.label_ids, predicted, labels=[0, 1])
    latency = measure_latency(model, tokenizer, raw["test"][0]["text"])
    peak_gpu = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None
    callback.sample_resources()
    ram_gib = callback.peak_rss / 1024**3
    with (output / "test_predictions.csv").open("w", encoding="utf-8", newline="") as f:
        f.write("# Sources: docs/data.md; docs/evaluation.md; docs/implementation.md R-07\n")
        writer = csv.writer(f)
        writer.writerow(["test_subset_index", "actual", "predicted"])
        writer.writerows(zip(range(len(predicted)), prediction.label_ids.tolist(), predicted.tolist()))
    result = {"config": asdict(config), "environment": env, "splits": splits,
              "model_revision": model_revision, "dataset_revision": dataset_revision,
              "training": {**training.metrics, "wall_seconds": train_seconds,
                            "total_parameters": sum(p.numel() for p in model.parameters()),
                            "gradient_norms": callback.gradient_norms},
              "epochs": callback.history, "validation_metrics": validation,
              "test_metrics": prediction.metrics, "confusion_matrix": matrix.tolist(),
              "classification_report": classification_report(prediction.label_ids, predicted,
                  labels=[0, 1], target_names=["NEGATIVE", "POSITIVE"], output_dict=True, zero_division=0),
              "latency": latency, "gpu_peak_allocated_gib": peak_gpu,
              "host_peak_observed_rss_gib": ram_gib, "demo": demo_results,
              "early_stopped": callback.early_stopped,
              "early_stopping_triggered": callback.policy.stale >= callback.policy.patience}
    quality = config.mode == "full"
    result["kpis"] = {
        "test_accuracy_ge_0.85": bool(prediction.metrics["test_accuracy"] >= .85) if quality else "N/A: non-full run",
        "test_macro_f1_ge_0.85": bool(prediction.metrics["test_f1_macro"] >= .85) if quality else "N/A: non-full run",
        "validation_accuracy_ge_0.85": bool(validation["validation_accuracy"] >= .85) if quality else "N/A: non-full run",
        "latency_mean_le_25ms": latency["mean_ms"] <= 25,
        "gpu_peak_le_4GiB": peak_gpu <= 4 if peak_gpu is not None else "N/A: CPU",
        "host_observed_rss_le_6GiB": ram_gib <= 6,
        "T4_epoch_le_180s": (max(x["epoch_wall_seconds"] for x in callback.history) <= 180)
            if "T4" in env["device"] else "N/A: device is not Colab T4"}
    write_json(output / "metrics.json", result)
    plot_results(output, trainer.state.log_history, matrix)
    render_report(output, result)
    print("RESULT " + json.dumps({"test": prediction.metrics, "kpis": result["kpis"]}), flush=True)
    return result


## 9. Smoke test trong kernel

Notebook thực hiện smoke ở `outputs/notebook-smoke/` để kiểm chứng đường chạy trực tiếp trong kernel, gồm gradient, save/reload và báo cáo. Nếu run này đã có, kiểm hash trước khi tái sử dụng. Run mới phải dùng thư mục mới/rỗng; không ghi đè bằng chứng.

Hàm thực thi ghi thêm snapshot notebook vào manifest. Cấu hình artifact phải khớp cấu hình yêu cầu; thay tham số cần chọn thư mục run mới.

In [ ]:
# Sources: Lab 3/docs/implementation.md; Lab 3/docs/training.md; Lab 3/docs/AI_Usage.md
def ensure_run(config, destination):
    destination = Path(destination)
    if (destination/'metrics.json').is_file():
        verify_artifacts(destination)
        measured = json.loads((destination/'metrics.json').read_text(encoding='utf-8'))
        if measured['config'] != asdict(config):
            raise ValueError('Cấu hình run đã lưu khác cấu hình yêu cầu; chọn thư mục output mới.')
        print('Tái sử dụng run đã kiểm hash:', destination.name)
        return measured
    destination = fresh_directory(destination)
    try:
        with (destination/'run.log').open('w',encoding='utf-8') as log:
            with contextlib.redirect_stdout(Tee(sys.stdout,log)):
                print('Sources: '+'; '.join(SOURCES))
                measured = train_run(config,destination)
    except Exception as exc:
        write_json(destination/'failure.json',{'exception':type(exc).__name__,'message':str(exc)})
        raise
    finally:
        shutil.copyfile(NOTEBOOK_PATH,destination/'notebook_source.ipynb')
        support.record_provenance(destination,f'Notebook: train_run(mode={config.mode})')
        manifest_path = destination/'provenance.json'
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        manifest['execution_entrypoint'] = NOTEBOOK_PATH.name
        manifest['notebook_source_sha256'] = sha256(destination/'notebook_source.ipynb')
        with manifest_path.open('w',encoding='utf-8',newline=chr(10)) as f:
            f.write(json.dumps(manifest,ensure_ascii=False,indent=2)+chr(10))
    verify_artifacts(destination)
    return measured

SMOKE_DIR = ROOT/'outputs/notebook-smoke'
smoke = ensure_run(RunConfig(mode='smoke'),SMOKE_DIR)
assert smoke['epochs'][-1]['step'] == 2
assert all(math.isfinite(x) for x in smoke['training']['gradient_norms'])
assert any(x>0 for x in smoke['training']['gradient_norms'])
show_table(['Kiểm tra smoke','Kết quả'],[
    ['Optimizer steps',smoke['epochs'][-1]['step']],
    ['Gradient hữu hạn và khác 0','PASS'],
    ['Lưu/nạp lại checkpoint và sinh báo cáo','PASS'],
    ['KPI chất lượng','N/A: dữ liệu rút gọn']])


## 10. Full training và kết quả độc lập

`outputs/full/` chứa full run đã thực thi trên GPU với 3 epochs và 3.750 optimizer steps. Notebook kiểm hash, đối chiếu cấu hình và nạp số đo thực; không train lại chỉ để thay đổi điểm test. Nếu chưa có run, chính hàm `train_run` vừa định nghĩa trong notebook sẽ thực thi full.

In [ ]:
# Sources: Lab 3/docs/implementation.md; Lab 3/docs/training.md; Lab 3/docs/evaluation.md
RUN_DIR = ROOT/'outputs/full'
results = ensure_run(CONFIG,RUN_DIR)
show_table(['Split','Số mẫu','NEGATIVE','POSITIVE'],
           [(name,info['size'],*info['class_counts']) for name,info in results['splits'].items()])
show_table(['Chỉ số','Giá trị'],[
    ['Validation accuracy',f"{100*results['validation_metrics']['validation_accuracy']:.2f}%"],
    ['Test accuracy',f"{100*results['test_metrics']['test_accuracy']:.3f}%"],
    ['Test macro F1',f"{results['test_metrics']['test_f1_macro']:.6f}"],
    ['Test binary F1',f"{results['test_metrics']['test_f1_binary']:.6f}"],
    ['Test precision',f"{results['test_metrics']['test_precision']:.6f}"],
    ['Test recall',f"{results['test_metrics']['test_recall']:.6f}"],
    ['Thiết bị',results['environment']['device']]])


## 11. Tiến trình, tài nguyên và tiêu chí nghiệm thu

Thời gian epoch gồm train/eval/save; tổng train wall không bao gồm tải model/dữ liệu và test cuối. GPU allocation là số PyTorch ghi nhận; RSS host là đỉnh quan sát tại callback. Không dùng tốc độ RTX để kết luận tiêu chí dành riêng cho Colab T4.

In [ ]:
# Sources: Lab 3/docs/evaluation.md; Lab 3/docs/implementation.md
show_table(['Epoch','Val loss','Val accuracy','Macro F1','Wall (s)'],[
    [f"{row['epoch']:.0f}",f"{row['val_loss']:.6f}",f"{100*row['val_accuracy']:.2f}%",
     f"{row['val_f1_macro']:.6f}",f"{row['epoch_wall_seconds']:.2f}"] for row in results['epochs']])
show_table(['Tài nguyên / thời gian','Giá trị'],[
    ['Train wall',f"{results['training']['wall_seconds']:.2f} s"],
    ['Latency mean / p95',f"{results['latency']['mean_ms']:.3f} / {results['latency']['p95_ms']:.3f} ms"],
    ['Input latency',f"{results['latency']['token_count']} tokens; batch 1"],
    ['GPU peak allocated',f"{results['gpu_peak_allocated_gib']:.3f} GiB"],
    ['Host observed RSS',f"{results['host_peak_observed_rss_gib']:.3f} GiB"],
    ['Điều kiện early stopping kích hoạt',results.get('early_stopping_triggered',False)],
    ['Lịch thực sự bị rút ngắn',results['early_stopped']]])
show_table(['Tiêu chí','Kết quả'],[
    [key,('PASS' if value else 'FAIL') if isinstance(value,bool) else value]
    for key,value in results['kpis'].items()])


## 12. Biểu đồ và nhận xét

Loss train/validation lấy từ cùng run; confusion matrix chỉ tập test. Train loss giảm trong khi validation loss tăng sau epoch 1 là dấu hiệu overfitting và tăng tự tin ở dự đoán sai. Validation accuracy chỉ cải thiện nhẹ.

Theo chính sách xác định trước, `best` chọn epoch có validation accuracy cao nhất; `best_val_loss` giữ epoch khác nếu cần. Không thay chính sách sau khi xem test. Confusion matrix có hàng nhãn thật, cột nhãn dự đoán; NEG/POS tương ứng 0/1.

In [ ]:
# Sources: Lab 3/docs/evaluation.md; Lab 3/docs/verification.md
display(Markdown('### Training và validation loss'))
display(Image(filename=str(RUN_DIR/'loss_curve.png')))
display(Markdown('### Confusion matrix — tập test độc lập'))
display(Image(filename=str(RUN_DIR/'confusion_matrix.png')))
correct = sum(results['confusion_matrix'][i][i] for i in range(2))
total = results['splits']['test']['size']
print(f'Dự đoán đúng: {correct:,}/{total:,} ({100*correct/total:.3f}%)')


## 13. Demo checkpoint đã fine-tune

Nạp lại `checkpoints/best` từ đĩa, không phụ thuộc model còn trong RAM. Module B được huấn luyện trên tiếng Anh. Với câu trung tính, model vẫn bắt buộc chọn một trong hai lớp; không diễn giải score như độ chắc chắn tuyệt đối.

In [ ]:
# Sources: Lab 3/docs/demo.md; Lab 3/docs/model.md
demo_model,demo_tokenizer,predictions = load_demo(RUN_DIR/'checkpoints/best',DEMO_TEXTS)
show_table(['Văn bản','Nhãn','Score'],[
    [row['text'],row['label'],f"{row['score']:.6f}"] for row in predictions])


## 14. Kiểm thử và truy nguồn

Các kiểm thử dùng tiny model ngẫu nhiên chỉ xác minh cơ chế; không đóng góp vào số đo chất lượng IMDB. Phần kiểm tra dưới xác nhận hai cách tính F1, đầy đủ nguồn Markdown, artifact nguyên vẹn và bộ test của module dùng chung.

In [ ]:
# Sources: Lab 3/docs/verification.md; Lab 3/docs/traceability.md; Lab 3/docs/AI_Usage.md
known = compute_metrics((np.array([[2.,0.]]*4),np.array([0,0,0,1])))
assert known['accuracy'] == .75 and known['f1_binary'] == 0
assert abs(known['f1_macro']-3/7)<1e-12
subprocess.run([sys.executable,str(ROOT/'test.py')],check=True)
source_audit = support.audit_sources()
show_table(['Kiểm chứng','Kết quả'],[
    ['Định nghĩa cốt lõi khớp AST',f'{len(matched_symbols)} / {len(matched_symbols)}'],
    ['Source Markdown',source_audit['status']],
    ['Số ô code có nguồn',source_audit['notebook_code_cells_checked']],
    ['Artifact Module A',verify_artifacts(A_DIR)['status']],
    ['Artifact smoke trong kernel',verify_artifacts(SMOKE_DIR)['status']],
    ['Artifact full',verify_artifacts(RUN_DIR)['status']],
    ['Known-label metrics','PASS']])


## 15. Kết luận và giới hạn

Full run đạt test accuracy **91,012%** và macro F1 **0,910118** trên 25.000 mẫu test, vượt mục tiêu nội bộ 85%. Giá trị chi tiết được lấy từ artifact đã kiểm chứng ở các bảng phía trên.

- Validation/test tách riêng; chỉ validation quyết định checkpoint.
- Có dấu hiệu overfitting ở các epoch sau. Cần dùng validation cho nghiên cứu regularization hoặc số epoch ở thực nghiệm tiếp theo.
- Module B chỉ có hai lớp và huấn luyện trên tiếng Anh; không có lớp trung tính.
- Chưa đo tốc độ trên Colab T4; RSS là đỉnh quan sát, không phải đỉnh tuyệt đối.
- Checkpoint phục vụ suy luận; không kèm trạng thái optimizer để resume.

## Nguồn và phạm vi

- [Báo cáo kiểm chứng](Lab%203/docs/verification.md), [đánh giá](Lab%203/docs/evaluation.md), [training](Lab%203/docs/training.md).
- [Nguồn artifact full](Lab%203/outputs/full/artifacts.md), [báo cáo full](Lab%203/outputs/full/report.md).
- [Khai báo AI](Lab%203/docs/AI_Usage.md): ghi nhận đóng góp và trách nhiệm kiểm chứng; [truy vết](Lab%203/docs/traceability.md): yêu cầu → triển khai → kiểm thử.